# 07 Multimodal Models

This notebook defines the deep multimodal modeling setup for early sepsis prediction. It aligns structured trajectories with temporal note windows, instantiates multiple structured encoders and fusion strategies, and prepares experiment-ready artifacts for full training.

## Architectural plan

- Structured encoder options: GRU or Transformer.
- Text encoder path: Clinical transformer embeddings aggregated across windows.
- Fusion strategies: early fusion, late fusion, gated fusion, cross-modal attention.
- This notebook prepares aligned multimodal examples and validates model wiring; full training loops can be expanded in later experiments or scripts.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy torch transformers

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.models.multimodal import MultimodalClassifier
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config['multimodal']

## Load structured and text artifacts

In [ ]:
horizon = int(config['prediction']['horizons_hours'][0])
dataset_name = f'horizon_{horizon}h'
structured_path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
text_path = paths['processed_data_dir'] / '05_text_processing' / f'{dataset_name}_note_windows.csv'
assert structured_path.exists(), f'Missing structured dataset: {dataset_name}. Run 04_feature_engineering first.'
assert text_path.exists(), f'Missing note-window dataset: {dataset_name}_note_windows. Run 05_text_processing first.'

structured_df = pd.read_csv(structured_path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'])
text_df = pd.read_csv(text_path, parse_dates=['prediction_time', 'first_note_time', 'last_note_time'])
structured_df.shape, text_df.shape

## Build aligned multimodal examples

For now, we keep this stage lightweight: structured inputs are grouped into fixed-length sequences and note windows receive placeholder embedding tensors. In a full run, those text embeddings should come from ClinicalBERT or a similar encoder.

In [ ]:
id_cols = ['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID']
feature_cols = [
    col for col in structured_df.columns
    if ('__' in col or col == 'hours_since_icu_admit') and pd.api.types.is_numeric_dtype(structured_df[col])
]

stay_index = structured_df[id_cols + ['split', 'sepsis3_label']].drop_duplicates().copy()
stay_index = stay_index.merge(
    text_df[id_cols].drop_duplicates().assign(has_text=1),
    on=id_cols,
    how='left',
)
stay_index['has_text'] = stay_index['has_text'].fillna(0).astype(int)
stay_index.head()

In [ ]:
example_stays = stay_index.head(4)
structured_batch = []
text_batch = []
labels = []

for _, stay in example_stays.iterrows():
    stay_rows = structured_df.loc[(structured_df['ICUSTAY_ID'] == stay['ICUSTAY_ID'])].sort_values('hour')
    seq = stay_rows[feature_cols].fillna(0.0).tail(config['multimodal']['max_structured_steps']).to_numpy(dtype='float32')
    if len(seq) < config['multimodal']['max_structured_steps']:
        pad = np.zeros((config['multimodal']['max_structured_steps'] - len(seq), len(feature_cols)), dtype='float32')
        seq = np.vstack([pad, seq])

    stay_notes = text_df.loc[(text_df['ICUSTAY_ID'] == stay['ICUSTAY_ID'])].sort_values('note_window_index')
    n_windows = min(len(stay_notes), config['multimodal']['max_text_windows'])
    text_embeddings = np.zeros((config['multimodal']['max_text_windows'], config['multimodal']['text_embedding_dim']), dtype='float32')
    if n_windows > 0:
        # Placeholder embeddings for wiring validation; replace with encoder outputs in a full run.
        text_embeddings[:n_windows] = np.random.normal(0, 0.02, size=(n_windows, config['multimodal']['text_embedding_dim'])).astype('float32')

    structured_batch.append(seq)
    text_batch.append(text_embeddings)
    labels.append(int(stay['sepsis3_label']))

structured_tensor = torch.tensor(np.stack(structured_batch))
text_tensor = torch.tensor(np.stack(text_batch))
label_tensor = torch.tensor(labels, dtype=torch.float32)
structured_tensor.shape, text_tensor.shape, label_tensor.shape

## Instantiate fusion variants and run a dry forward pass

In [ ]:
experiment_rows = []
artifact_bundle = {}

for fusion_strategy in config['multimodal']['fusion_strategies']:
    model = MultimodalClassifier(
        structured_input_dim=len(feature_cols),
        text_embedding_dim=config['multimodal']['text_embedding_dim'],
        hidden_dim=config['multimodal']['hidden_dim'],
        dropout=config['multimodal']['dropout'],
        structured_encoder_type=config['multimodal']['structured_encoder'],
        fusion_strategy=fusion_strategy,
    )
    logits, aux = model(structured_tensor, text_tensor)
    probs = torch.sigmoid(logits).detach().cpu().numpy()
    experiment_rows.append({
        'dataset_name': dataset_name,
        'structured_encoder': config['multimodal']['structured_encoder'],
        'fusion_strategy': fusion_strategy,
        'batch_size': int(structured_tensor.shape[0]),
        'structured_input_dim': int(structured_tensor.shape[-1]),
        'text_windows': int(text_tensor.shape[1]),
        'dry_run_mean_probability': float(probs.mean()),
    })

    dry_run_df = example_stays[id_cols + ['split', 'sepsis3_label']].copy()
    dry_run_df['fusion_strategy'] = fusion_strategy
    dry_run_df['dry_run_probability'] = probs
    artifact_bundle[f'{dataset_name}_{fusion_strategy}_dry_run'] = dry_run_df

experiment_plan_df = pd.DataFrame(experiment_rows)
experiment_plan_df

## Save experiment plan artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '07_multimodal_models'
artifact_bundle['multimodal_experiment_plan'] = experiment_plan_df
artifact_bundle['multimodal_example_stays'] = example_stays
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '07_multimodal_models_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='07_multimodal_models',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'dataset_name': dataset_name,
        'fusion_strategies': config['multimodal']['fusion_strategies'],
        'preferred_text_model': config['text_processing']['pretrained_text_model_name'],
    },
)
manifest_path

## Review checklist

Before clinical evaluation, review:

- Are the structured and text examples aligned on the same ICU stays?
- Which fusion strategies should get full training first?
- Does the selected text encoder fit the note window lengths?
- Should the structured encoder remain GRU or switch to Transformer for the main experiments?
- Are additional batching or masking utilities needed before large-scale training?

## Next notebook

`08_evaluation.ipynb` will consolidate model outputs into discrimination, calibration, and clinical utility metrics and create the paper-ready evaluation artifacts.